# KlimSta Monte Carlo Analysis

This notebook analyses Monte Carlo results for one version of the KlimSta game model.

The main goals are to:

1. inspect whether the simulated game outcomes look plausible,
2. identify cards that are associated with better final performance,
3. distinguish common cards from cards that are disproportionately represented in good games,
4. account for whether cards were actually available to the player,
5. test whether results depend strongly on the chosen definition of an "elite" game.

## Important interpretation

`vp` denotes **Victory Points / CO₂ score**.  
**Lower VP is better.**

The card metrics in this notebook are primarily **descriptive associations**. In particular, `vp_lift` does not by itself prove that a card causally improves the outcome. A card may also be associated with better games because it is chosen in favourable states or as part of a successful combination of measures.


## 1. Setup

The notebook assumes it is run from the **repository root**.

Expected files:

```text
Versionen/paper_draft_v1/
├── game_data.xlsx
└── results/
    ├── games.parquet
    └── plays.parquet
```

`plays.parquet` is only available when the simulation was run with `log_choices=True`.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from model.game import load_game_data
from model.analysis import (
    game_summary,
    strategy_summary,
    card_analysis,
    card_elite_sensitivity,
    card_occurrence,
    card_elite_occurrence,
    card_enrichment,
    card_vp_lift,
    card_round_summary,
    card_opportunity_summary,
    plot_game_distributions,
    plot_vp_distribution,
    plot_state_vs_vp,
    plot_card_enrichment,
    plot_card_vp_lift,
)

version = Path("Versionen/paper_draft_v1")
results_dir = version / "results"

game_data = load_game_data(version)

games = pd.read_parquet(results_dir / "games.parquet")

plays_path = results_dir / "plays.parquet"
plays = pd.read_parquet(plays_path) if plays_path.exists() else pd.DataFrame()

print(f"{len(games):,} games loaded")
print(f"{len(plays):,} logged card decisions loaded")


## 2. Basic simulation sanity checks

Before interpreting individual cards, inspect the overall simulation.

Useful questions:

- Is the VP range plausible?
- Do games normally survive all four rounds?
- Are final budget values plausible?
- Do thermal protection, electricity demand, generation, storage, and satisfaction occupy reasonable parts of their tracks?
- Are there obvious pile-ups at minimum or maximum values that could indicate clamping or lookup-table problems?


In [ ]:
game_summary(games)

In [ ]:
plot_game_distributions(games)
plt.show()

In [ ]:
plot_vp_distribution(games, elite_share=0.05)
plt.show()

### State variables versus VP

These plots are useful for checking whether the direction of the model behaves as expected.

For example, if higher thermal protection generally leads to lower VP, this should be visible here. These plots are descriptive and do not control for interactions with other cards.


In [ ]:
for variable in [
    "thermal_protection",
    "electricity_generation",
    "electricity_demand",
    "storage",
    "satisfaction",
]:
    if variable in games.columns:
        plot_state_vs_vp(games, variable)
        plt.show()

## 3. Combined card-effectiveness table

The main card-level table combines several complementary metrics.

### Occurrence

`occurrence` is the fraction of all games in which a card was played.

### Elite occurrence

`elite_occurrence` is the fraction of the best games containing the card.  
By default, the best **5%** of games are used.

### Enrichment

\[
\text{enrichment} =
\frac{P(\text{card} \mid \text{elite game})}
     {P(\text{card} \mid \text{all games})}
\]

Interpretation:

- `1.0`: card is equally common in elite and ordinary games
- `> 1.0`: card is overrepresented in elite games
- `< 1.0`: card is underrepresented in elite games

### VP lift

\[
\text{VP lift} =
\overline{VP}_{\text{without card}}
-
\overline{VP}_{\text{with card}}
\]

Because lower VP is better:

- **positive VP lift = associated with better games**
- negative VP lift = associated with worse games

The confidence interval is a descriptive normal-approximation interval around the difference in means.

### Selection rate

When detailed choice logging is available:

\[
\text{selection rate} =
\frac{\text{times selected}}
     {\text{times playable}}
\]

This helps distinguish a card that is rarely chosen because it is rarely available from one that is frequently available but usually ignored.


In [ ]:
cards = card_analysis(
    games,
    plays=plays,
    game_data=game_data,
    elite_share=0.05,
)

cards

## 4. Which cards are most strongly associated with good games?

Sorting by `vp_lift` answers:

> In this simulation, which cards occur in games with the largest difference in final VP compared with games where the card was not played?

This should not yet be interpreted as a causal card value.


In [ ]:
columns = [
    "card",
    "label_en",
    "category",
    "occurrence",
    "elite_occurrence",
    "enrichment",
    "vp_lift",
    "vp_lift_ci_low",
    "vp_lift_ci_high",
]

columns = [c for c in columns if c in cards.columns]

cards.sort_values("vp_lift", ascending=False)[columns].head(20)

In [ ]:
plot_card_vp_lift(cards, top_n=30)
plt.show()

## 5. Which cards are disproportionately represented in elite games?

This comparison separates **frequency** from **success association**.

A card can be very common overall but not especially important for elite games. Conversely, a rare card can have a high enrichment ratio if it appears disproportionately often among the best-performing games.

In the scatter plot:

- the diagonal represents equal occurrence in all and elite games,
- cards above the diagonal are overrepresented among elite games,
- cards below the diagonal are underrepresented.


In [ ]:
cards.sort_values("enrichment", ascending=False)[columns].head(20)

In [ ]:
plot_card_enrichment(
    cards,
    min_occurrence=0.02,
)
plt.show()

## 6. Opportunity-adjusted card selection

This section requires `plays.parquet`, generated with:

```python
run_simulation(
    version,
    ...,
    log_choices=True,
)
```

The analysis counts a logical card only once per decision, even if several physical copies of that card were present.

This is useful for answering:

> When the card was actually playable, how often was it selected?


In [ ]:
if plays.empty:
    print("No detailed play log available. Re-run the simulation with log_choices=True.")
else:
    opportunities = card_opportunity_summary(plays)

    opportunity_view = cards.merge(
        opportunities,
        on="card",
        how="left",
        suffixes=("", "_check"),
    )

    columns = [
        "card",
        "label_en",
        "playable_opportunities",
        "selected",
        "selection_rate",
        "vp_lift",
        "enrichment",
    ]

    columns = [c for c in columns if c in opportunity_view.columns]

    display(
        opportunity_view
        .sort_values("selection_rate", ascending=False)[columns]
        .head(30)
    )

## 7. When are cards played?

Timing can help distinguish:

- **foundation cards** that are normally selected early,
- **late optimisation cards**,
- cards that only become available after prerequisites are met.

This section also requires `log_choices=True`.


In [ ]:
if plays.empty:
    print("No detailed play log available.")
else:
    timing = card_round_summary(plays)

    timing = timing.merge(
        cards[["card", "label_en"]].drop_duplicates(),
        on="card",
        how="left",
    )

    timing.sort_values(
        ["median_round", "times_played"],
        ascending=[True, False],
    ).head(40)

## 8. Sensitivity to the elite-game cutoff

A card should not be called important solely because it performs well under one arbitrary cutoff.

The following table repeats the enrichment analysis for the best:

- 1%
- 5%
- 10%

A robust card should generally remain positively enriched across several cutoffs.


In [ ]:
sensitivity = card_elite_sensitivity(
    games,
    elite_shares=(0.01, 0.05, 0.10),
)

card_info = cards[
    ["card", "label_en", "category"]
].drop_duplicates()

sensitivity = sensitivity.merge(
    card_info,
    on="card",
    how="left",
)

sensitivity.head()

In [ ]:
sensitivity_pivot = sensitivity.pivot(
    index=["card", "label_en"],
    columns="elite_share",
    values="enrichment",
)

sensitivity_pivot.columns = [
    f"enrichment_{share:.0%}"
    for share in sensitivity_pivot.columns
]

sensitivity_pivot.sort_values(
    "enrichment_5%",
    ascending=False,
).head(30)

## 9. Candidate cards for closer inspection

For the paper, the most interesting cards are likely those where several indicators point in the same direction.

A strong candidate would typically show:

- positive VP lift,
- enrichment above 1,
- sufficient occurrence to avoid being driven by a handful of games,
- and, where logged, a meaningful selection rate.

The following is only a convenient screening table, not a statistical decision rule.


In [ ]:
candidate_cards = cards.copy()

candidate_cards = candidate_cards.loc[
    (candidate_cards["occurrence"] >= 0.02)
    & (candidate_cards["enrichment"] > 1)
    & (candidate_cards["vp_lift"] > 0)
]

candidate_columns = [
    "card",
    "label_en",
    "category",
    "occurrence",
    "elite_occurrence",
    "enrichment",
    "vp_lift",
    "vp_lift_ci_low",
    "vp_lift_ci_high",
    "median_round",
    "selection_rate",
]

candidate_columns = [
    c for c in candidate_columns
    if c in candidate_cards.columns
]

candidate_cards[
    candidate_columns
].sort_values(
    ["enrichment", "vp_lift"],
    ascending=False,
)

## 10. Strategy comparison

This becomes relevant once additional strategies such as `cheapskate`, `collector`, `electrician`, `insulator`, or `user_first` are simulated.

If all strategies are stored together in `games`, this table compares their final outcomes directly.


In [ ]:
if "strategy" in games.columns and games["strategy"].nunique() > 1:
    display(strategy_summary(games))
else:
    print("Only one strategy is currently present in the simulation results.")

## 11. Interpretation checklist for the paper

Before calling a card "effective", check whether the following agree:

1. **Occurrence:** Is there enough simulation exposure to estimate anything reliably?
2. **Elite enrichment:** Is the card overrepresented among good games?
3. **VP lift:** Are games containing the card associated with lower final VP?
4. **Confidence interval:** Is the estimated association reasonably precise?
5. **Opportunity:** Was the card actually available often enough?
6. **Selection rate:** Do simulated strategies choose it when it is playable?
7. **Timing:** Is its apparent value dependent on early or late play?
8. **Sensitivity:** Does the result persist for 1%, 5%, and 10% elite cutoffs?
9. **Strategy dependence:** Does the conclusion persist across different player strategies?

Only after these checks should pairwise interactions or more advanced counterfactual card-value analysis be added.
